# Stage 4 — End-to-end 파이프라인 (PDF → view → element → Donut → JSON)

도면 PDF 1장을 넣으면 전체 파이프라인을 통과시켜 구조화 JSON 을 조립합니다.

**큰 그림**: 도면은 한 번에 읽기 어렵습니다. 그래서 **"점점 좁혀가며 본다"** 는 전략을 씁니다.
도면 전체 → 뷰(부분 도면) → 요소(치수·기호) 순으로 잘라 들어간 뒤, 마지막에 요소 안의
**글자(값)** 를 읽습니다. 즉 **"어디에 있나(검출)"는 YOLO**, **"뭐라고 적혔나(인식)"는 Donut** 이
나눠 맡습니다.

```
┌──────────────────────────────────────────────────────────────────────┐
│  입력:  도면 PDF 1장                                                    │
└──────────────────────────────────────────────────────────────────────┘
        │
        │ ① 래스터화  (PyMuPDF · 300 DPI)
        │    벡터 PDF → 픽셀 이미지 1장. 이후 단계는 모두 이미지를 봄
        ▼
   page.png  ── 도면 전체 한 장
        │
        │ ② 뷰 검출  ──  YOLO (detect, AABB = 수평 사각형 박스)
        │    "도면 안에 정면도·평면도·표제란 같은 '뷰'가 어디 있나?"
        │    뷰는 똑바로 놓여 있어 회전 없는 박스(AABB)로 충분
        ▼
   뷰 크롭들  ── 예: 5개  (정면도 / 측면도 / 표제란 …)
        │
        │   ┌─ 각 뷰 크롭마다 ③~④ 반복 ───────────────────────────┐
        │   │                                                       │
        │   │ ③ 요소 검출  ──  YOLO (OBB = 회전 가능한 박스) + rectify
        │   │    "이 뷰 안에 치수·공차·거칠기 같은 '요소'가 어디 있나?"
        │   │    치수선은 기울어 있기도 함 → OBB 로 잡아
        │   │    똑바로 펴서(rectify) 잘라냄 → Donut 이 읽기 쉬워짐
        │   │      ▼
        │   │   요소 크롭들 (+ type)
        │   │      ── 크롭마다 YOLO 가 매긴 종류(dimension / gdt / … )
        │   │      │
        │   │ ④ 요소 인식  ──  Donut  (크롭 1장 → 텍스트 값)
        │   │    "이 요소 크롭에 적힌 값이 정확히 뭐냐?"
        │   │    예) "Ø65" ,  "Ra 1.6" ,  "M8" …
        │   │      ▼
        │   │   {type, value, box}  ── 종류 + 읽은 값 + 위치
        │   └───────────────────────────────────────────────────────┘
        │
        │ ⑤ 조립(assemble)
        │    뷰별로 요소들을 묶고, 요소 좌표를 뷰 기준 → page(전체) 기준으로 환산
        ▼
┌──────────────────────────────────────────────────────────────────────┐
│  출력:  구조화 JSON                                                     │
│  {                                                                     │
│    "views": [                                                          │
│      { "view_index": 0,                                                │
│        "box": [...],                       ← 뷰의 page 상 위치          │
│        "elements": [                                                   │
│          {"type": "dimension",      "value": "Ø65",    "box": [...]},  │
│          {"type": "surface_finish", "value": "Ra 1.6", "box": [...]}   │
│        ] }                                                             │
│    ]                                                                   │
│  }                                                                     │
└──────────────────────────────────────────────────────────────────────┘
```

> **용어**
> - **AABB** (Axis-Aligned Bounding Box): 회전 없는 수평 직사각형 박스. 뷰처럼 똑바로 놓인 영역에 사용.
> - **OBB** (Oriented Bounding Box): 기울기(각도)를 가진 박스. 회전된 치수선·기호를 정확히 감쌈.
> - **rectify**: OBB 로 잡은 기울어진 요소를 회전 보정해 **똑바로 편** 크롭. 글자가 수평이라 인식 정확도↑.
> - **검출(YOLO) vs 인식(Donut)**: YOLO 는 "**어디에** 무엇이 있나(위치+종류)", Donut 은 "거기 **뭐라고** 적혔나(값)".

**커널**: 전 과정을 **단일 커널 `kardi_env`** 로 실행합니다. `kardi_env` 에 YOLO(ultralytics)와
Donut(transformers/torch) 의존성이 모두 설치돼 있어 **커널 전환이 필요 없습니다**.
- **Part A**: 검출·크롭 → `data/_pipeline_tmp/` 에 크롭 + `meta.json` 저장
- **Part B**: 크롭을 Donut 으로 읽어 값 채우고 최종 JSON 조립

→ Part A → Part B 를 같은 커널에서 위→아래로 이어서 실행하면 됩니다.
(`meta.json`/크롭은 디버깅용 중간 산출물.)

## Part A — 검출·크롭 (커널: **kardi_env**)

In [1]:
# ── 설정 ─────────────────────────────────────────────────────────
from pathlib import Path
import json, sys
sys.path.append("detection")                     # crop_utils
from crop_utils import save_crops_from_result
from ultralytics import YOLO

PDF_PATH   = Path("../data/raw_pdf/A1370954730.pdf")  # ← held-out 도면으로 바꾸세요
TMP        = Path("../data/_pipeline_tmp"); TMP.mkdir(parents=True, exist_ok=True)
# ultralytics 8.4.67 이 task 를 runs/{task}/runs/{name} 으로 중첩 저장함 (train_*.ipynb 산출물)
VIEW_PT    = "detection/view/runs/detect/runs/view/weights/best.pt"      # view: detect task
ELEM_PT    = "detection/element/runs/obb/runs/element/weights/best.pt"   # element: obb task
DPI        = 300
VIEW_PAD   = 4     # view 크롭 여유 픽셀 (element→page 좌표 변환에 사용)
ELEM_PAD   = 2     # element 정렬 크롭 여유 픽셀

In [2]:
# ── 1) PDF → page.png ────────────────────────────────────────────
import fitz
doc = fitz.open(PDF_PATH)
pix = doc[0].get_pixmap(matrix=fitz.Matrix(DPI/72, DPI/72), alpha=False)
page_png = TMP / f"{PDF_PATH.stem}.png"
pix.save(page_png); doc.close()
print("page:", page_png)

page: ../data/_pipeline_tmp/A1370954730.png


In [3]:
# ── 2) view 검출 → view 크롭 ─────────────────────────────────────
# train_view.ipynb 에서 학습한 모델(VIEW_PT : best.pt) 사용
view_model = YOLO(VIEW_PT)
vr = view_model.predict(page_png, imgsz=1280, conf=0.25, verbose=False)[0]
view_meta = save_crops_from_result(vr, TMP / "views", PDF_PATH.stem, pad=VIEW_PAD)
print(f"view {len(view_meta)}개")

view 5개


In [4]:
# ── 3) 각 view 크롭 → element 검출 → rectify 크롭 ────────────────
elem_model = YOLO(ELEM_PT)  # 요소 YOLO 모델 로드

def _to_page(xy, ox, oy):
    # element box(view 기준) → page 기준 좌표로 변환. OBB, AABB 모두 지원
    if xy and isinstance(xy[0], (list, tuple)):  # OBB polygon
        return [[float(x) + ox, float(y) + oy] for x, y in xy]
    return [xy[0] + ox, xy[1] + oy, xy[2] + ox, xy[3] + oy]  # AABB

records = []   # 뷰/요소 메타데이터를 저장
for vi, vm in enumerate(view_meta):
    er = elem_model.predict(vm["path"], imgsz=1024, conf=0.25, verbose=False)[0]  # 요소 검출
    elems = save_crops_from_result(
        er,
        TMP / "elements" / f"view{vi}",
        f"v{vi}",
        pad=ELEM_PAD
    )  # 요소 크롭 저장

    # view 크롭의 page 상 기준점 (pad까지 고려, 음수 방지)
    ox, oy = max(0.0, vm["xy"][0] - VIEW_PAD), max(0.0, vm["xy"][1] - VIEW_PAD)
    for e in elems:
        e["xy_page"] = _to_page(e["xy"], ox, oy)  # 요소 box를 page 좌표로 변환
    records.append({"view_index": vi, "view": vm, "elements": elems})

meta_path = TMP / "meta.json"
json.dump(records, open(meta_path, "w", encoding="utf-8"), ensure_ascii=False, indent=2)  # 결과 저장
n_elem = sum(len(r["elements"]) for r in records)
print(f"element 총 {n_elem}개 → {meta_path}")  # 총 요소 개수 출력

element 총 47개 → ../data/_pipeline_tmp/meta.json


## Part B — Donut 값 추론 + JSON 조립 (커널: **kardi_env**)

### 🔑 `decode_donut_output(seq, proc.tokenizer, TASK)` — 모델 출력 → 구조화 dict

Donut 의 `generate()` 가 내놓는 건 사람이 바로 쓰기 좋은 JSON 이 아니라
**특수 토큰이 잔뜩 섞인 한 줄짜리 문자열**입니다. 예:

```
<s_element><s_dimension>Ø65</s_dimension></s>
```

이 함수는 이 문자열을 깔끔한 dict(`{"dimension": "Ø65"}`)로 바꿔주는 **"마지막 번역기"** 이며, 딱 2단계로 동작합니다.

**인자 3개**

| 인자 | 역할 |
|------|------|
| `seq` | 모델이 생성한 원본 문자열 (위 예시처럼 토큰 범벅) |
| `proc.tokenizer` | `eos`/`pad`/`bos` 같은 **특수 토큰이 뭔지** 알아내려고 넘김 |
| `TASK` (`<s_element>`) | 맨 앞에 붙는 **task 토큰**도 떼어내야 해서 넘김 |

**동작 2단계**

1. **청소 (특수·task 토큰 제거)** — `eos`/`pad`/`bos` 와 task 토큰(`<s_element>`)을 지웁니다.
   ```
   <s_element><s_dimension>Ø65</s_dimension></s>  →  <s_dimension>Ø65</s_dimension>
   ```
   - ⚠️ **task 토큰을 안 떼면 망가집니다**: `<s_element>` 는 **닫힘 짝(`</s_element>`)이 없는** 여는 태그라,
     다음 단계의 정규식이 "닫힘 없는 키"로 오인 → 파싱이 깨지고 **점수가 0** 이 됩니다.
     (그래서 strip 대상 토큰 집합을 이 헬퍼 한 곳에서 관리)

2. **파싱 (`token2json`)** — 남은 `<s_키>값</s_키>` 패턴을 정규식으로 재귀 파싱:
   ```
   <s_dimension>Ø65</s_dimension>  →  {"dimension": "Ø65"}
   ```
   값 안에 또 태그가 중첩되면 재귀로 파고들고, 풀 구조가 없으면 문자열을 그대로 반환.

> **한 줄 요약**: 모델 출력 토큰열 → `{"<타입>": "<값>"}` dict 로 되돌리는 함수.
> 학습 때 dict 를 토큰으로 바꿨던 `json2token` 의 **정확한 역과정**이고,
> 이 dict 가 바로 다음 조립 셀에서 `type`/`value` 로 풀려 최종 JSON 에 들어갑니다.

In [5]:
# ── Donut 로드 + 추론 함수 ───────────────────────────────────────
import json, sys, torch
from pathlib import Path
from PIL import Image
from transformers import DonutProcessor, VisionEncoderDecoderModel
sys.path.append("detection")                     # 공용 토큰 I/O (학습 노트북과 동일 로직)
from donut_utils import decode_donut_output

TMP        = Path("../data/_pipeline_tmp")
CKPT       = "../checkpoints_elements/final"
TASK       = "<s_element>"
device     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
proc       = DonutProcessor.from_pretrained(CKPT, use_fast=False)
emodel     = VisionEncoderDecoderModel.from_pretrained(CKPT).to(device).eval()

@torch.inference_mode()
def read_value(img_path):
    # 주의: DonutProcessor는 입력 이미지를 반드시 RGB (3채널)로 기대하므로, 흑백/투명(1, L, LA, P, RGBA 등) 이미지도 강제 변환 필요
    image = Image.open(img_path).convert("RGB")  # .convert("RGB")를 하지 않으면 모델 입력 shape 오류나 알파채널 처리 문제 발생 가능(필수)
    
    # proc(image, return_tensors="pt")는 이미지를 토크나이저 및 이미지 프로세싱을 거쳐 PyTorch 텐서 형태로 반환합니다.
    # .pixel_values는 전처리된 이미지 텐서를 가져오고, .to(device)는 해당 텐서를 GPU나 CPU에 올립니다.
    pv = proc(image, return_tensors="pt").pixel_values.to(device)
    
    # TASK 프롬프트("<s_element>")만 토큰화해서 디코더 입력(dec)으로 사용 (특수토큰 제외)
    dec = proc.tokenizer(TASK, add_special_tokens=False, return_tensors="pt").input_ids.to(device)
    
   
    # emodel.generate()로 이미지와 태스크 프롬프트(dec)로부터 답변 토큰(out) 생성
    out = emodel.generate(
        pv,
        decoder_input_ids=dec,
        max_new_tokens=128,
        pad_token_id=proc.tokenizer.pad_token_id,
        eos_token_id=proc.tokenizer.eos_token_id,
        use_cache=True
    )
                   
    # out 텐서(모델이 생성한 토큰 IDs)를 사람이 읽을 수 있는 문자열 시퀀스로 디코딩합니다.
    # skip_special_tokens=False라서 eos, pad 등 특수 토큰까지 모두 출력합니다.
    seq = proc.batch_decode(out, skip_special_tokens=False)[0]
    
    # 모델 출력 토큰열 → {"<타입>": "<값>"} dict 로 되돌리는 함수. 학습 때 dict를 토큰으로 바꿨던 json2token의 정확한 역과정이고, 
    # 이 dict가 바로 다음 조립 셀에서 type/value로 풀려 최종 JSON에 들어갑니다.
    return decode_donut_output(seq, proc.tokenizer, TASK)

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


### 🧩 조립(assemble) — 크롭별 값 추론 결과를 최종 JSON 한 덩어리로

지금까지 흩어져 있던 조각들을 **하나의 구조화 JSON 으로 합치는 마지막 단계**입니다.
재료는 두 가지뿐:
- **`meta.json`** — Part A 의 YOLO 가 남긴 **위치·종류** 정보 (뷰 박스, 요소 박스, 요소 type, 크롭 경로)
- **`read_value()`** — 요소 크롭 1장을 Donut 으로 읽어 **값**을 돌려주는 함수 (바로 위 셀)

즉 **"어디에·무슨 종류(YOLO)"** + **"뭐라고 적혔나(Donut)"** 를 요소마다 짝지어 붙입니다.

**동작 흐름 (2중 루프)**

```
meta.json 의 뷰들 ──┐
  각 뷰 r 마다       │  view_out = {view_index, box(=뷰의 page 위치), elements: []}
    각 요소 e 마다   │    parsed = read_value(e["path"])   ← Donut 추론 (크롭 → 값)
                     │    value  = (보정 후 최종 값)
                     │    elements 에 {type, value, conf, box, box_in_view} append
  └──────────────────┘  → assembled["views"] 에 누적
```

**핵심 한 줄 — type 보정 (`value` 결정)**

`read_value()` 가 돌려주는 dict 는 학습 스키마에 따라 두 형태일 수 있습니다.
**YOLO 가 분류한 type(`e["name"]`)** 을 키로 시도하되, 없으면 dict 전체를 값으로 보존합니다:

```python
value = parsed.get(e["name"], parsed) if isinstance(parsed, dict) else parsed
```

| 상황 | 꺼내는 값 |
|------|-----------|
| ① **평면 스키마** dict 에 YOLO type 키 있음 (`{"Dimension":"Ø65"}`) | `parsed["Dimension"]` → `"Ø65"` |
| ② **구조화 스키마** dict, type 키 없음 (`{"nominalValue":"Ø65","quantity":"2"}`) | `parsed` **통째로** (구조 필드 유실 방지) |
| ③ 애초에 dict 가 아님 (파싱 실패한 raw 문자열) | `parsed` 문자열 그대로 |

> ⚠️ **수정 이력**: 과거 코드는 type 키가 없으면 `next(iter(parsed.values()))` 로 **첫 필드 값 하나만**
> 취해, 구조화 스키마(quantity·upperLimit·lowerLimit …)의 나머지 필드를 **조용히 버렸습니다**.
> 이제 dict 전체를 보존해 구조 정보 손실이 없습니다.
> 또한 기호 복원(`U+00D8`→`Ø`)은 `decode_donut_output` 안의 `decode_tree` 가 처리합니다.

**각 요소에 담기는 필드**

| 필드 | 의미 |
|------|------|
| `type` | YOLO 가 매긴 요소 종류 (`Dimension` / `GD&T_FCF` …) |
| `value` | Donut 이 읽은 값 (`"Ø65"` 또는 구조화 dict) |
| `conf` | YOLO 검출 신뢰도 |
| `box` | **page(전체) 좌표** — 뷰 박스와 같은 좌표계라 바로 겹쳐 그릴 수 있음 |
| `box_in_view` | (디버그용) 뷰 크롭 내부 좌표 |

> **한 줄 요약**: `meta.json`(위치+종류) × `read_value`(값) 를 요소마다 합쳐
> `{views:[{box, elements:[{type, value, box}]}]}` 형태의 **최종 `result.json`** 을 만들고 저장합니다.

In [ ]:
# ── 조립: meta.json 의 element 크롭마다 값 추론 → 최종 JSON ──────
records = json.load(open(TMP / "meta.json", encoding="utf-8"))
assembled = {"source": str(TMP), "views": []}
for r in records:
    view_out = {"view_index": r["view_index"], "box": r["view"]["xy"], "elements": []}
    for e in r["elements"]:
        parsed = read_value(e["path"])        # dict: 평면 {type:값} 또는 구조화 {nominalValue:…}
        # YOLO type 으로 보정: 평면 스키마면 그 키의 값을, 구조화 스키마면(키 없음) 구조 dict 통째로.
        # (예전처럼 next(iter(...)) 첫 값만 취하면 구조 필드가 조용히 유실되므로 dict 전체 보존)
        value = parsed.get(e["name"], parsed) if isinstance(parsed, dict) else parsed
        view_out["elements"].append({"type": e["name"], "value": value,
                                      "conf": round(e["conf"], 3),
                                      "box": e["xy_page"],       # 페이지 좌표 (view box 와 동일 좌표계)
                                      "box_in_view": e["xy"]})   # 디버그용: view-크롭 내부 좌표
    assembled["views"].append(view_out)

print(json.dumps(assembled, ensure_ascii=False, indent=2)[:2000])
out_path = TMP / "result.json"
json.dump(assembled, open(out_path, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print("\n저장:", out_path)